In [19]:
import pandas as pd

X = pd.read_csv("Xtillyet.csv")
y = pd.read_csv("yfinal.csv")
test_data = pd.read_csv("dftesttillyet.csv")

In [21]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")


In [22]:
test_ids = df_test["Id"].copy()

In [2]:
test_ids = pd.read_csv("test_ids.csv")

In [3]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from scipy.optimize import minimize

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [8]:
y_log = np.log1p(y)

In [4]:
xgb_model = XGBRegressor(
    n_estimators=1513,
    learning_rate=0.025062361836477583,
    max_depth=3,
    min_child_weight=1.6983893030290647,
    subsample=0.8603805272092565,
    colsample_bytree=0.744304731223331,
    gamma=0.0001579074785360672,
    reg_alpha=0.00011921318833886257,
    reg_lambda=0.8029281069659692,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


cat_model = CatBoostRegressor(
    iterations=4187,
    learning_rate=0.004109054956033522,
    depth=6,
    l2_leaf_reg=0.35658455151507984,
    random_strength=4.706979541014375,
    bagging_temperature=5.950159666491976,
    loss_function="RMSE",
    verbose=False,
    random_seed=42
)


lgbm_model = LGBMRegressor(
    n_estimators=3492,
    learning_rate=0.006844292325219354,
    num_leaves=36,
    max_depth=4,
    min_child_samples=21,
    subsample=0.5879917744659336,
    colsample_bytree=0.4625177254406786,
    reg_alpha=0.08153368237146785,
    reg_lambda=0.02675810935391093,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


gbr_model = GradientBoostingRegressor(
    n_estimators=3938,
    learning_rate=0.012261053653303986,
    max_depth=3,
    min_samples_split=19,
    min_samples_leaf=29,
    max_features=0.37327832037835607,
    loss="huber",
    random_state=42
)


elastic_model = ElasticNet(
    alpha=0.00014264008017096888,
    l1_ratio=0.390328026770505,
    max_iter=20000,
    random_state=42
)

In [5]:
models = {
    "XGB": xgb_model,
    "CAT": cat_model,
    "LGBM": lgbm_model,
    "GBR": gbr_model,
    "ELASTIC": elastic_model
}

In [6]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
n_models = len(models)

oof_preds = np.zeros((len(X), n_models))
test_preds = np.zeros((len(test_data), n_models))

In [8]:
for model_idx, (model_name, model) in enumerate(models.items()):

    print(f"\nTraining {model_name}")

    fold_test_preds = []
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y_log.iloc[train_idx]
        y_val = y_log.iloc[val_idx]

        fold_model = clone(model)

        fold_model.fit(
            X_train,
            y_train
        )

        # Validation predictions
        val_pred = fold_model.predict(X_val)

        # Store OOF prediction
        oof_preds[val_idx, model_idx] = val_pred

        # Fold score
        fold_rmse = root_mean_squared_error(
            y_val,
            val_pred
        )

        fold_scores.append(fold_rmse)

        print(
            f"Fold {fold}: "
            f"{fold_rmse:.6f}"
        )

        # Test prediction from this fold
        fold_test_pred = fold_model.predict(
            test_data
        )

        fold_test_preds.append(
            fold_test_pred
        )

    # Average test predictions from all folds
    test_preds[:, model_idx] = np.mean(
        fold_test_preds,
        axis=0
    )

    print(
        f"{model_name} mean CV: "
        f"{np.mean(fold_scores):.6f}"
    )


Training XGB
Fold 1: 0.130337
Fold 2: 0.115827
Fold 3: 0.165293
Fold 4: 0.125129
Fold 5: 0.109940
XGB mean CV: 0.129305

Training CAT
Fold 1: 0.129694
Fold 2: 0.115003
Fold 3: 0.146187
Fold 4: 0.121806
Fold 5: 0.105105
CAT mean CV: 0.123559

Training LGBM
Fold 1: 0.133067
Fold 2: 0.117768
Fold 3: 0.157518
Fold 4: 0.127770
Fold 5: 0.111031
LGBM mean CV: 0.129431

Training GBR


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\ensemble\_gb.py:691: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


Fold 1: 0.128675


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\ensemble\_gb.py:691: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)  # TODO: Is this still required?


Fold 2: 0.110576


...

c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.368847e+00, tolerance: 1.917e-02
  model = cd_fast.enet_coordinate_descent(


Fold 3: 0.228422


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.853104e+00, tolerance: 1.811e-02
  model = cd_fast.enet_coordinate_descent(


Fold 4: 0.156377
Fold 5: 0.119877
ELASTIC mean CV: 0.154449


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.426556e+00, tolerance: 1.932e-02
  model = cd_fast.enet_coordinate_descent(


In [9]:
print("\nOOF MODEL SCORES")
print("-" * 40)

for model_idx, model_name in enumerate(models.keys()):

    score = root_mean_squared_error(
        y_log,
        oof_preds[:, model_idx]
    )

    print(
        f"{model_name}: {score:.6f}"
    )


OOF MODEL SCORES
----------------------------------------
XGB: 0.130743
CAT: 0.124339
LGBM: 0.130416
GBR: 0.126716
ELASTIC: 0.159312


In [10]:
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
import numpy as np

def loss(w):

    # force weights to be positive and sum to 1
    w = np.abs(w)
    w = w / np.sum(w)

    # weighted combination of all 5 model OOF predictions
    blend = oof_preds @ w

    return np.sqrt(
        mean_squared_error(
            y_log,
            blend
        )
    )


res = minimize(
    loss,
    x0=[0.2] * 5,
    method="Nelder-Mead"
)

In [11]:
best_weights = np.abs(res.x)
best_weights = best_weights / best_weights.sum()

In [12]:
for name, weight in zip(models.keys(), best_weights):
    print(f"{name}: {weight:.6f}")

print(
    "OOF Ensemble RMSE:",
    loss(best_weights)
)

XGB: 0.000001
CAT: 0.652688
LGBM: 0.000031
GBR: 0.347280
ELASTIC: 0.000000
OOF Ensemble RMSE: 0.12356578805177562


In [4]:
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from mlxtend.regressor import StackingCVRegressor

In [5]:
kfolds = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
alphas_alt = [
    14.5, 14.6, 14.7, 14.8, 14.9,
    15, 15.1, 15.2, 15.3, 15.4, 15.5
]

alphas2 = [
    5e-05, 0.0001, 0.0002, 0.0003,
    0.0004, 0.0005, 0.0006,
    0.0007, 0.0008
]

e_alphas = [
    0.0001, 0.0002, 0.0003,
    0.0004, 0.0005, 0.0006,
    0.0007
]

e_l1ratio = [
    0.8, 0.85, 0.9, 0.95, 0.99, 1
]


ridge = make_pipeline(
    RobustScaler(),
    RidgeCV(
        alphas=alphas_alt,
        cv=kfolds
    )
)


lasso = make_pipeline(
    RobustScaler(),
    LassoCV(
        max_iter=10_000_000,
        alphas=alphas2,
        random_state=42,
        cv=kfolds
    )
)


elasticnet = make_pipeline(
    RobustScaler(),
    ElasticNetCV(
        max_iter=10_000_000,
        alphas=e_alphas,
        cv=kfolds,
        l1_ratio=e_l1ratio
    )
)


svr = make_pipeline(
    RobustScaler(),
    SVR(
        C=20,
        epsilon=0.008,
        gamma=0.0003
    )
)


gbr = GradientBoostingRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=4,
    max_features="sqrt",
    min_samples_leaf=15,
    min_samples_split=10,
    loss="huber",
    random_state=42
)


lightgbm = LGBMRegressor(
    objective="regression",
    num_leaves=4,
    learning_rate=0.01,
    n_estimators=5000,
    max_bin=200,
    bagging_fraction=0.75,
    bagging_freq=5,
    bagging_seed=7,
    feature_fraction=0.2,
    feature_fraction_seed=7,
    verbosity=-1
)


xgboost = XGBRegressor(
    learning_rate=0.01,
    n_estimators=3460,
    max_depth=3,
    min_child_weight=0,
    gamma=0,
    subsample=0.7,
    colsample_bytree=0.7,
    objective="reg:squarederror",
    n_jobs=-1,
    reg_alpha=0.00006,
    random_state=27
)

In [17]:
stack_gen = StackingCVRegressor(
    regressors=(
        ridge,
        lasso,
        elasticnet,
        svr,
        gbr,
        xgboost,
        lightgbm
    ),
    meta_regressor=XGBRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=2,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ),
    use_features_in_secondary=True,
    cv=5,
    shuffle=True,
    random_state=42
)

In [18]:
scores = cross_val_score(
    stack_gen,
    X,
    y_log,
    scoring="neg_root_mean_squared_error",
    cv=kfolds,
    n_jobs=-1
)

stack_score = -scores.mean()

print("Stack CV Log-RMSE:", stack_score)

Stack CV Log-RMSE: 0.1277512326836586


In [19]:
print(scores)
print("Mean CV Log-RMSE:", -scores.mean())
print("Std:", scores.std())

[-0.13337909 -0.11728875 -0.15487361 -0.1235894  -0.10962532]
Mean CV Log-RMSE: 0.1277512326836586
Std: 0.015637322853999356


In [ ]:
stack_gen.fit(X, y_log)

In [ ]:
stack_log_pred = stack_gen.predict(test_data)

In [9]:
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from sklearn.linear_model import (
    Ridge,
    RidgeCV,
    LassoCV,
    ElasticNetCV
)

from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from mlxtend.regressor import StackingCVRegressor


# ============================================================
# 1. CROSS-VALIDATION
# ============================================================

kfolds = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 2. RIDGE
# ============================================================

alphas_ridge = [
    14.5, 14.6, 14.7, 14.8, 14.9,
    15.0, 15.1, 15.2, 15.3, 15.4, 15.5
]

ridge = make_pipeline(
    RobustScaler(),
    RidgeCV(
        alphas=alphas_ridge,
        cv=kfolds
    )
)


# ============================================================
# 3. LASSO
# ============================================================

alphas_lasso = [
    5e-05,
    0.0001,
    0.0002,
    0.0003,
    0.0004,
    0.0005,
    0.0006,
    0.0007,
    0.0008
]

lasso = make_pipeline(
    RobustScaler(),
    LassoCV(
        max_iter=10_000_000,
        alphas=alphas_lasso,
        random_state=42,
        cv=kfolds
    )
)


# ============================================================
# 4. ELASTIC NET
# ============================================================

elastic_alphas = [
    0.0001,
    0.0002,
    0.0003,
    0.0004,
    0.0005,
    0.0006,
    0.0007
]

elastic_l1_ratios = [
    0.8,
    0.85,
    0.9,
    0.95,
    0.99,
    1.0
]

elasticnet = make_pipeline(
    RobustScaler(),
    ElasticNetCV(
        max_iter=10_000_000,
        alphas=elastic_alphas,
        l1_ratio=elastic_l1_ratios,
        cv=kfolds
    )
)


# ============================================================
# 5. SVR
# ============================================================

svr = make_pipeline(
    RobustScaler(),
    SVR(
        C=20,
        epsilon=0.008,
        gamma=0.0003
    )
)


# ============================================================
# 6. GBR
# ============================================================

gbr = GradientBoostingRegressor(
    n_estimators=3938,
    learning_rate=0.012261053653303986,
    max_depth=3,
    min_samples_split=19,
    min_samples_leaf=29,
    max_features=0.37327832037835607,
    loss="huber",
    random_state=42
)


# ============================================================
# 7. XGBOOST
# ============================================================

xgboost = XGBRegressor(
    n_estimators=1513,
    learning_rate=0.025062361836477583,
    max_depth=3,
    min_child_weight=1.6983893030290647,
    subsample=0.8603805272092565,
    colsample_bytree=0.744304731223331,
    gamma=0.0001579074785360672,
    reg_alpha=0.00011921318833886257,
    reg_lambda=0.8029281069659692,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 8. LIGHTGBM
# ============================================================

lightgbm = LGBMRegressor(
    n_estimators=3492,
    learning_rate=0.006844292325219354,
    num_leaves=36,
    max_depth=4,
    min_child_samples=21,
    subsample=0.5879917744659336,
    colsample_bytree=0.4625177254406786,
    reg_alpha=0.08153368237146785,
    reg_lambda=0.02675810935391093,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)


# ============================================================
# 9. CATBOOST
# ============================================================

catboost = CatBoostRegressor(
    iterations=4187,
    learning_rate=0.004109054956033522,
    depth=6,
    l2_leaf_reg=0.35658455151507984,
    random_strength=4.706979541014375,
    bagging_temperature=5.950159666491976,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)


# ============================================================
# 10. META MODEL
# ============================================================

meta_model = Ridge(
    alpha=10.0
)


# ============================================================
# 11. STACKING MODEL
# ============================================================

stack_gen = StackingCVRegressor(
    regressors=(
        ridge,
        lasso,
        elasticnet,
        svr,
        gbr,
        xgboost,
        lightgbm,
        catboost
    ),

    meta_regressor=meta_model,

    use_features_in_secondary=False,

    cv=5,
    shuffle=True,
    random_state=42,

    n_jobs=-1
)


# ============================================================
# 12. EVALUATE STACK
# ============================================================

scores = cross_val_score(
    stack_gen,
    X,
    y_log,
    scoring="neg_root_mean_squared_error",
    cv=kfolds,
    n_jobs=-1
)

print("\nStack fold scores:")
print(-scores)

stack_score = -scores.mean()

print(
    "\nMean Stack CV Log-RMSE:",
    stack_score
)

print(
    "Stack CV Standard Deviation:",
    scores.std()
)


# ============================================================
# 13. EVALUATE CATBOOST ALONE
# ============================================================

cat_scores = cross_val_score(
    catboost,
    X,
    y_log,
    scoring="neg_root_mean_squared_error",
    cv=kfolds,
    n_jobs=-1
)

cat_score = -cat_scores.mean()

print("\nCatBoost fold scores:")
print(-cat_scores)

print(
    "\nCatBoost Mean CV Log-RMSE:",
    cat_score
)

print(
    "CatBoost CV Standard Deviation:",
    cat_scores.std()
)


# ============================================================
# 14. COMPARISON
# ============================================================

print("\n----------------------------------")
print("FINAL COMPARISON")
print("----------------------------------")

print(f"CatBoost: {cat_score:.6f}")
print(f"Stack:    {stack_score:.6f}")
print(f"Difference: {cat_score - stack_score:.6f}")

if stack_score < cat_score:
    print("\nStack is better.")
else:
    print("\nCatBoost is better.")


# ============================================================
# 15. TRAIN FINAL STACK
# ============================================================

stack_gen.fit(
    X,
    y_log
)


# ============================================================
# 16. PREDICT TEST DATA
# ============================================================

stack_log_pred = stack_gen.predict(
    test_data
)

stack_pred = np.expm1(
    stack_log_pred
)


print("\nFinished.")
print("stack_log_pred shape:", stack_log_pred.shape)
print("stack_pred shape:", stack_pred.shape)


Stack fold scores:
[0.12766438 0.11096757 0.1765836  0.12556379 0.10452288]

Mean Stack CV Log-RMSE: 0.1290604451933081
Stack CV Standard Deviation: 0.025306403187031066

CatBoost fold scores:
[0.12969408 0.11500253 0.14618697 0.12180634 0.10510451]

CatBoost Mean CV Log-RMSE: 0.12355888574071094
CatBoost CV Standard Deviation: 0.013903364724049683

----------------------------------
FINAL COMPARISON
----------------------------------
CatBoost: 0.123559
Stack:    0.129060
Difference: -0.005502

CatBoost is better.


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn


Finished.
stack_log_pred shape: (1459,)
stack_pred shape: (1459,)


c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


In [13]:
y_log = np.asarray(y_log).ravel()

In [14]:
stack_gen.fit(X, y_log)

stack_log_pred = stack_gen.predict(test_data)

print(stack_log_pred.shape)
# should be (1459,)

stack_pred = np.expm1(stack_log_pred)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": stack_pred
})

submission.to_csv(
    "stack_submission.csv",
    index=False
)

c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but RobustScaler was fitted without feature names
  warnings.warn(
c:\Users\ayush\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but GradientBoostingRegress

(1459,)


ValueError: Data must be 1-dimensional, got ndarray of shape (1459, 2) instead

In [15]:
print(stack_pred.shape)
print(test_ids.shape)

(1459,)
(1459, 2)


In [17]:
print("y_log shape:", np.asarray(y_log).shape)
print("stack_log_pred shape:", np.asarray(stack_log_pred).shape)
print("first predictions:")
print(stack_log_pred[:5])

y_log shape: (1460,)
stack_log_pred shape: (1459,)
first predictions:
[11.71422061 11.98843605 12.10862684 12.16939842 12.13710489]


In [18]:
print("test_ids shape:", np.asarray(test_ids).shape)

stack_pred = np.expm1(stack_log_pred)

print("stack_pred shape:", stack_pred.shape)

test_ids shape: (1459, 2)
stack_pred shape: (1459,)


In [16]:
stack_pred_final = stack_pred.mean(axis=1)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": stack_pred_final
})

submission.to_csv(
    "stack_submission.csv",
    index=False
)

AxisError: axis 1 is out of bounds for array of dimension 1

In [23]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": stack_pred
})

submission.to_csv(
    "stack_submission.csv",
    index=False
)